# Notebook para el calculo de ratios financieras



A partir del panel limpio, este notebook calcula los ratios financieros (liquidez,
endeudamiento, rentabilidad y eficiencia) y el Z''-Score de Altman, que define la
zona de salud de cada empresa. Luego arma la etiqueta temporal: empareja los ratios
del año *t* con la salud del año *t+1*, para poder anticipar el deterioro con un año
de anticipación.

Entrada: `panel_2018_2024.csv` · Resultado: `panel_enriquecido.csv`

In [1]:
#configuracion para poder importar desde src
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.ratios import calcular_ratios

# Cargar el panel limpio que guardó 01_carga
panel = pd.read_csv("../data/processed/panel_2018_2024.csv")

# Calcular los ratios con la función del módulo
panel = calcular_ratios(panel)

panel[["razon_corriente", "endeudamiento", "apalancamiento", "roa",
       "margen_neto", "margen_operacional", "rotacion_activos"]].describe()

,razon_corriente,endeudamiento,apalancamiento,roa,margen_neto,margen_operacional,rotacion_activos
count,2.008700e+04,20109.000000,2.010900e+04,20109.000000,1.869800e+04,18698.000000,19109.000000
mean,7.565369e+02,5.521519,-3.926324e+02,2.111328,-7.549749e+01,-55.624431,12.735478
std,7.104677e+04,619.978680,5.629688e+04,308.734924,7.577922e+03,5241.500949,1599.399428
min,0.000000e+00,0.000000,-7.983212e+06,-481.825106,-1.024338e+06,-703207.000000,0.000000
25%,1.068575e+00,0.327273,3.349713e-01,-0.008747,-5.862654e-03,0.014842,0.320849
50%,1.584456e+00,0.562716,1.000823e+00,0.021624,3.070125e-02,0.070542,0.888186
75%,2.762632e+00,0.787095,2.530569e+00,0.072524,9.095579e-02,0.154277,1.457927
max,8.990994e+06,87785.000000,7.291635e+03,43777.000000,8.290674e+03,7260.627604,221094.000000


Razón corriente: mediana 1.58, con el rango típico entre 1.07 y 2.76. Es normal, la mayoría de empresas tiene entre 1 y 2.7 veces sus pasivos corrientes en activos corrientes.
Endeudamiento: mediana 0.56 (financian el 56% de sus activos con deuda), rango 0.33–0.79. 
ROA: mediana 2.2%, rango −0.9% a 7.3%. Rentabilidades modestas pero realistas.
Margen neto: mediana 3.1%. Margen operacional: mediana 7.0%. Rotación de activos: mediana 0.89. Todos en rangos perfectamente sensatos.

Parecen haber grandes outliers

En src/altman.py se calcula el Z´´-Score.
Fórmula que combina cuatro ratios en un solo número que mide la probabilidad de que una empresa caiga en dificultades financieras. Se utiliza la versión para mercados emergentes (Altman, 1995):

Z'' = 6.56·X1 + 3.26·X2 + 6.72·X3 + 1.05·X4

Los cuatro componentes, y por qué cada uno predice el deterioro:

X1 = (Activo corriente − Pasivo corriente) / Activo total : el capital de trabajo sobre los activos. Mide liquidez: una empresa con capital de trabajo negativo no puede cubrir sus obligaciones inmediatas

X2 = Ganancias acumuladas / Activo total : la rentabilidad acumulada en el tiempo. Bajo o negativo señala una empresa joven o que ha venido perdiendo dinero

X3 = Utilidad operacional (EBIT) / Activo total : la rentabilidad operativa, el poder de generar ganancias con el negocio. Es el término con más peso (6.72), porque la capacidad de generar utilidades es el mejor predictor de supervivencia

X4 = Patrimonio / Pasivo total : refleja la solvencia, cuánto colchón de patrimonio hay frente a la deuda


El número resultante se traduce en una categoría de salud:

Z'' > 2.6 → zona segura (bajo riesgo)

1.1 ≤ Z'' ≤ 2.6 → zona gris (incierta)

Z'' < 1.1 → zona de riesgo (alta probabilidad de dificultades)


Detalle importante: no aparecen las ventas.

La versión del Z´´-score para mercados emergentes elimina el término de ventas/activos del Z-score original. Por tal razón, las empresas que se observo previamente que no tienen declarados los ingresos (en 01_carga.ipynb) no seran un problema. El Z´´ no las necesita.

In [2]:
from src.altman import calcular_zscore, clasificar_zona

# Calcular el Z''-Score
panel = calcular_zscore(panel)

# Clasificar cada empresa en su zona de "salud"
panel["zona"] = panel["zscore"].apply(clasificar_zona)

# Ver la distribución de la salud financiera
panel["zona"].value_counts()

zona
segura    11862
riesgo     5021
gris       3212
Name: count, dtype: int64

Hay desbalance pero moderado. La zona "riesgo" es la minoría pero con 5.021 casos se tienen bastantes ejemplos de empresas en problemas para que el modelo aprenda. Aun asi se tratará como un desbalance del modelo porque "segura" domina.

In [3]:
# Validación de que la etiqueta tiene sentido en comparación a la columna "estado" de los archivos caratulas: estado legal de la empresa
import pandas as pd
pd.crosstab(panel["estado"], panel["zona"], normalize="index").round(3)

zona,gris,riesgo,segura
estado,,,
ACTIVA,0.160,0.241,0.598
ACUERDO DE REESTRUCTURACIÓN,0.188,0.444,0.368
ACUERDO DE REORGANIZACION,0.154,0.513,0.333
CONCORDATO EN EJECUCIÓN,0.000,1.000,0.000
EN ETAPA PREOPERATIVA,0.041,0.439,0.520


ACTIVA: 24% en riesgo 

ACUERDO DE REESTRUCTURACIÓN: 44% en riesgo

ACUERDO DE REORGANIZACIÓN: 51% en riesgo

Las empresas en procesos legales de insolvencia tienen el doble o más de probabilidad de caer en la zona de riesgo del Z'' que las empresas activas. La etiqueta construida solo con números financieros tiene sentido con la realidad de las empresas.

Los ratios de ETAPA PREOPERATIVA parecen poco fiables así que se exluirá al modelar.

Hasta ahora cada fila tiene la salud de una empresa en su propio año. Para poder modelar y predecir se usaran los ratios de hoy para anticipar la salud del año que viene. Para cada empresa-año se necesita pegarle como objetivo su zona de salud del año siguiente.

Nota: anio=año

In [4]:
# Tabla auxiliar: la salud de cada empresa-año reetiquetada al año anterior
futuro = panel[["nit", "anio", "zona"]].copy()
futuro = futuro.rename(columns={"zona": "zona_futura"})
futuro["anio"] = futuro["anio"] - 1

# Unimion de salud en t+1
panel = panel.merge(futuro, on=["nit", "anio"], how="left")

In [5]:
print("Filas totales:", len(panel))

panel_modelo = panel.dropna(subset=["zona_futura"])
print("Filas con etiqueta futura (listas para modelar):", len(panel_modelo))

# Veamos la distribución del objetivo
panel_modelo["zona_futura"].value_counts()

Filas totales: 20109
Filas con etiqueta futura (listas para modelar): 15877


zona_futura
segura    9550
riesgo    3796
gris      2531
Name: count, dtype: int64

In [7]:
panel.to_csv("../data/processed/panel_enriquecido.csv", index=False)

15.877 filas listas para modelar (de 20.109; Se pierde los 4.232 que no tienen año siguiente, todo 2024 más las empresas con na).

Ahora cada una de esas 15.877 filas es una transición temporal: los ratios de una empresa en un año, emparejados con su salud al año siguiente.